# DeepWhale End-to-End Pipeline

This notebook runs the full sequence:
1) Collect raw whale tx data
2) Build address features
3) Label addresses
4) Cluster profiles
5) Train classifier
6) Detect anomalies
7) Quick sanity previews

Prereqs: `.env` with `ALCHEMY_URL`, and `pip install -r requirements.txt` in this env. Run cells top-to-bottom.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

# Project root is one level up from notebooks/
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd()
    if not (BASE_DIR / "src").exists():
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR    = BASE_DIR / "data"
MODELS_DIR  = BASE_DIR / "models"
FIGURES_DIR = BASE_DIR / "data"

RAW_CSV       = DATA_DIR / raw / "raw_whale_transactions.csv"
FEATURES_CSV  = DATA_DIR / processed / "address_features.csv"
LABELED_CSV   = DATA_DIR / processed / "labeled_addresses.csv"
CLUSTERED_CSV = DATA_DIR / processed / "clustered_addresses.csv"
ANOMALY_CSV   = DATA_DIR / processed / "anomaly_scores.csv"
MODEL_PKL     = MODELS_DIR / "whale_classifier.pkl"

# ── Import pipeline modules ───────────────────────────────────────────────────
from src import data_collection
from src import feature_engineering
from src import labeling
from src import clustering
from src import classification
from src import anomaly_detection

print(f"Project root : {BASE_DIR}")
print(f"Data dir     : {DATA_DIR}")
print("All modules imported successfully.")

Project root : c:\D\UCU\ML\project
Data dir     : c:\D\UCU\ML\project\data
All modules imported successfully.


## 1) Collect raw whale transactions
Runs data_collection.py (uses ALCHEMY_URL). Adjust blocks inside the script if needed.

In [ ]:
# Requires ALCHEMY_URL in .env — comment out if you already have raw data
try:
    df_raw = data_collection.run(
        output_csv=RAW_CSV,
        num_blocks=500,
    )
    print(f"Raw transactions collected: {len(df_raw)}")
except Exception as e:
    print(f"[data_collection] Skipped or failed: {e}")

## 2) Feature engineering
Aggregates raw transactions to per-address features.

In [9]:
try:
    df_features = feature_engineering.run(
        raw_csv=RAW_CSV,
        features_csv=FEATURES_CSV,
    )
    print(f"Feature matrix: {df_features.shape}")
except Exception as e:
    print(f"[feature_engineering] Failed: {e}")

[feature_engineering] Failed: c:\D\UCU\ML\project\data\raw_whale_transactions.csv not found. Run data_collection first.


## 3) Labeling
Rule-based labels for each address.

In [ ]:
try:
    df_labeled = labeling.run(
        features_csv=FEATURES_CSV,
        labeled_csv=LABELED_CSV,
    )
    print(df_labeled["label"].value_counts().to_string())
except Exception as e:
    print(f"[labeling] Failed: {e}")

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe labeling.py
Loading features from c:\D\UCU\ML\project\data_new\address_features.csv ...
  3274 addresses

Label distribution:
  unknown_whale          3017 ( 92.2%)  avg_confidence=0.30
  accumulator             213 (  6.5%)  avg_confidence=0.74
  active_trader            44 (  1.3%)  avg_confidence=0.79

Saved labeled data to c:\D\UCU\ML\project\data_new\labeled_addresses.csv

--- active_trader (sample) ---
                                   address  tx_count_out  exchange_ratio  active_days  net_flow_eth  unique_receivers         label  label_confidence
0x0003B5Aa5E30E97FcC596BB5D0F3A75255E08D4e            17             0.0           10    12471.8346                10 active_trader            0.6875
0x0D0707963952f2fBA59dD06f2b425ace40b492Fe            21             0.0           18      406.2447                15 active_trader            0.9375

--- unknown_whale (sample) ---
                                   address  tx_count_out  e

## 4) Clustering
Unsupervised profiling (KMeans + DBSCAN) and plots.

In [ ]:
try:
    df_clustered = clustering.run(
        features_csv=FEATURES_CSV,
        output_csv=CLUSTERED_CSV,
        figures_dir=FIGURES_DIR,
        labeled_csv=LABELED_CSV,
    )
    print(df_clustered["kmeans_cluster"].value_counts().sort_index().to_string())
except Exception as e:
    print(f"[clustering] Failed: {e}")

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe clustering.py
Loaded 3274 addresses from labeled_addresses.csv

[KMeans] Finding best k ...
  k=2  silhouette=0.511
  k=3  silhouette=0.524
  k=4  silhouette=0.476
  k=5  silhouette=0.352
  k=6  silhouette=0.364
  k=7  silhouette=0.370
  k=8  silhouette=0.308
Best silhouette at k=3 (0.524), but k=4 (0.476) is within 9.2% â€” choosing k=4 for richer segmentation

[PCA] Plotting KMeans clusters ...
  Saved cluster_pca_kmeans.png
[t-SNE] Plotting KMeans clusters (may take a moment) ...
  Saved cluster_tsne_kmeans.png
[Radar] Plotting cluster behaviour profiles ...
  Saved cluster_radar.png

=== Cluster Profiles (medians) ===
                tx_count_out  total_eth_out  avg_tx_eth  unique_receivers  exchange_ratio  round_number_ratio  active_days  net_flow_eth
kmeans_cluster                                                                                                                          
0                        1.5         58.860      

## 5) Classification
Train whale classifier (XGBoost + RandomForest) and save bundle.

In [ ]:
try:
    metrics = classification.run(
        clustered_csv=CLUSTERED_CSV,
        model_pkl=MODEL_PKL,
        figures_dir=FIGURES_DIR,
    )
    print(f"XGBoost F1-macro: {metrics['f1_xgb']:.4f}")
    print(f"RF      F1-macro: {metrics['f1_rf']:.4f}")
except Exception as e:
    print(f"[classification] Failed: {e}")

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe classification.py
Loaded 6643 labeled addresses
  Kept 6638 (dropped 5 low-confidence)
  Train: 5310  |  Test: 1328
Classes: ['accumulator', 'active_trader', 'exchange_depositor', 'unknown_whale']

[XGBoost] Training ...
  F1 macro (XGBoost): 1.0000
                    precision    recall  f1-score   support

       accumulator       1.00      1.00      1.00       531
     active_trader       1.00      1.00      1.00         1
exchange_depositor       1.00      1.00      1.00        85
     unknown_whale       1.00      1.00      1.00       711

          accuracy                           1.00      1328
         macro avg       1.00      1.00      1.00      1328
      weighted avg       1.00      1.00      1.00      1328

  Saved cm_xgboost.png
  Saved feature_importance.png
  Computing SHAP values ...
  Saved shap_summary.png

[Random Forest] Training ...
  F1 macro (Random Forest): 0.7498
                    precision    recall  f1-scor

## 6) Anomaly detection
Isolation Forest on address features + timeline plot (needs raw + features).

In [ ]:
try:
    df_anomaly = anomaly_detection.run(
        features_csv=FEATURES_CSV,
        anomaly_csv=ANOMALY_CSV,
        figures_dir=FIGURES_DIR,
        raw_csv=RAW_CSV,
        clustered_csv=CLUSTERED_CSV,
        labeled_csv=LABELED_CSV,
    )
    n_anomalies = (df_anomaly["anomaly_label"] == -1).sum()
    print(f"Anomalous addresses: {n_anomalies} / {len(df_anomaly)}")
except Exception as e:
    print(f"[anomaly_detection] Failed: {e}")

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe anomaly_detection.py
Loading address features ...
  6643 addresses

[Isolation Forest] Detecting anomalous whale addresses ...
  Anomalous addresses: 665 / 6643 (10.0%)
  Saved anomaly_scores_plot.png

=== Top 10 Most Anomalous Whale Addresses ===
                                   address  anomaly_score  tx_count_out  total_eth_out  exchange_ratio  unique_receivers
0x28C6c06298d514Db089934071355E5743bf21d60      -0.808602           270    396221.8877          0.5520               131
0xf30ba13e4b04Ce5dC4D254Ae5FA95477800F0EB0      -0.802878           303    118317.1494          0.0000               147
0xA9Ac43f5b5e38155A288d1A01D2cbc4478E14573      -0.800478           287    110670.4522          0.0000               109
0x3606f0F14828cBF4962A284a4bFF93Bc94b63665      -0.795222             7     56854.7255          0.0000                 2
0xE69f81b825d7Dc31ee9becef4DbEab5cf30e3Abb      -0.783559             6     47140.0000          0.89

## 7) Quick sanity previews
Show tail/head of key outputs if they exist.

In [ ]:
outputs = {
    "raw"       : RAW_CSV,
    "features"  : FEATURES_CSV,
    "labeled"   : LABELED_CSV,
    "clustered" : CLUSTERED_CSV,
    "anomaly"   : ANOMALY_CSV,
    "model"     : MODEL_PKL,
}

for name, path in outputs.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        if path.suffix == ".csv":
            df = pd.read_csv(path)
            print(f"[OK] {name}: {path.name}  ({size_kb:.1f} KB, {len(df)} rows x {df.shape[1]} cols)")
        else:
            print(f"[OK] {name}: {path.name}  ({size_kb:.1f} KB)")
    else:
        print(f"[MISSING] {name}: expected at {path}")